# Chapter 17: LLM-as-Judge
## Topic 5: Risks of LLM Judging LLM + How to Sanity-Check the Judge

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every prior topic in this chapter has built toward and used a genuine, working judge mechanism — but has repeatedly, deliberately flagged one specific concern without yet addressing it directly: the judge itself is an LLM, subject to exactly the same reliability limitations as the system it's evaluating. This closing topic addresses that concern head-on: what specifically goes wrong when an LLM judges LLM-produced output, and what concrete, practical steps actually mitigate this risk, rather than simply naming it as a caveat and moving on.
- The core risk this topic names precisely: an LLM judge can be swayed by the same surface-level qualities that make a response *seem* good — confident tone, fluent language, plausible-sounding structure — without those qualities actually correlating with the deeper property (genuine reasoning coherence, genuine escalation appropriateness) the judge is supposed to be assessing. This is structurally the same "confident but wrong" risk Chapter 8 Topic 4 established for the original agent's own outputs, now shown to apply equally to the judge evaluating those outputs — a judge is not somehow immune to the failure modes it exists to detect in others.
- This connects directly and precisely to a concern already raised, but not yet resolved, at multiple earlier points in this notebook series: Chapter 9 Topic 10's warning about sanity-checking RAGAS's LLM-judged metrics against hand-built alternatives, Chapter 14 Topic 1's own note that "a judge model can itself be wrong," and this chapter's own Topic 2 explicitly flagging that "the judge's own assessment is itself a judgment call, not an infallible ground truth." This topic is where that repeatedly-deferred concern finally receives its own dedicated, concrete treatment.


### 2. Internal Working — Step by Step

**The specific, named risks of LLM-judging-LLM, and concrete mitigations for each:**

1. **Self-preference bias**: a judge model may systematically favor outputs that resemble its own characteristic style or reasoning patterns, independent of genuine quality — particularly acute if the same model serves as both the original agent and the judge (exactly Topic 2's own "correlated blind spots" concern). **Mitigation**: use a genuinely different, independent model for judging where the stakes justify the added complexity, and validate the judge's verdicts specifically for signs of style-based rather than substance-based preference.
2. **Surface-plausibility bias**: a judge can be swayed by confident tone, fluent phrasing, or plausible-sounding structure, mistaking these surface qualities for the genuine substance being assessed — exactly the risk underlying Topic 1's Case 2 example, where confused reasoning still reads fluently. **Mitigation**: Topic 3's decomposed rubric criteria, checking specific, semantic properties (fact-grounding, logical connection) rather than a single holistic "does this sound good" question, directly reduces this risk by forcing the judge's attention onto substantive, checkable properties rather than an overall, more easily-swayed impression.
3. **Position and framing sensitivity**: a judge's verdict can shift based on how a case is presented to it — the order information appears in, incidental wording choices in the judge prompt itself — reflecting the same prompt-sensitivity concern Chapter 9 Topic 3 and Chapter 10 Topic 4 already measured concretely for other kinds of model prompts. **Mitigation**: test the judge's verdict consistency across superficial variations in how the same underlying case is presented, exactly the kind of empirical validation this notebook series has repeatedly required for any prompt-dependent behavior.
4. **Inconsistency across repeated calls**: since LLM outputs are inherently probabilistic (Topic 3's own explicit caveat), the same judge, given the exact same case, may not produce the exact same verdict every single time it's called — this needs to be measured empirically, not assumed away, and a judge's practical reliability depends on this consistency being adequate for the decisions its verdicts will inform.


### 3. How This Is Implemented in This Project

- This project's judge mechanism (Topics 2-4) should be sanity-checked using exactly the same evidence-based validation discipline this notebook series has applied to every other model-dependent behavior: a labeled set of cases with known-good and known-bad reasoning traces or escalation decisions (extending Topic 1's own two illustrative cases into a larger, more comprehensive validation set), checking that the judge's verdicts genuinely align with that known ground truth before trusting it on real, ambiguous cases where no such ground truth exists.
- Chapter 9 Topic 10's own recommendation to cross-validate RAGAS's LLM-judged metrics against this project's hand-built alternatives is directly reusable here: this project's judge verdicts (Topics 2-4) should periodically be cross-checked against independent human review on a sample of flagged cases, specifically investigating any disagreement to determine whether the judge or the human reviewer's initial assessment was actually correct, rather than assuming either is automatically right.
- Given this project's real, demonstrated cost concerns for judge calls (Topic 2, Topic 4's cost-tiering discussion), a practical, ongoing sanity-checking practice should itself be tiered: a smaller, periodic validation sample (checking judge-verdict-vs-human-review agreement) rather than validating every single judge call the project ever makes, mirroring exactly the same cost-conscious sampling discipline already established for Chapter 9 Topic 7's hallucination-rate monitoring.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Trusting a judge's verdict without any sanity-checking reproduces exactly the "trusting a model output without verification" mistake this entire notebook series has repeatedly warned against** (Chapter 8 Topic 4's hallucination-detection motivation, Chapter 9 Topic 10's RAGAS cross-validation recommendation) — a judge verdict is a model output like any other, deserving the same skepticism and validation discipline as any other model-generated artifact this project produces.
- **Self-preference bias is a particularly acute, easily-overlooked risk when convenience drives the decision to use the same model for both the original task and judging** — the correlated-blind-spot concern Topic 2 raised isn't hypothetical; it's a documented phenomenon in LLM-as-judge research generally, and this project's own judge-design decisions should account for it explicitly rather than defaulting to same-model convenience without considering this specific risk.
- **A judge that's overly lenient (passing most cases regardless of genuine quality) is arguably more dangerous than one that's overly strict**, since an overly lenient judge provides false confidence that reasoning-quality or escalation-appropriateness problems are being caught when they aren't — sanity-checking specifically needs to test the judge's ability to correctly *fail* known-bad cases, not just its ability to correctly *pass* known-good ones, since a judge that passes everything trivially "succeeds" on the good-case half of validation while providing zero real value.
- **Debugging a judge that seems to be behaving inconsistently or unreliably requires the same layered-attribution discipline as any other agent behavior in this notebook series** — is the inconsistency in the rubric's own criteria (Topic 3's concern, if the criteria are ambiguous), in the judge prompt's specific wording (a Chapter 10 Topic 4-style schema/prompt-clarity issue), or in the fundamental, irreducible probabilistic variability of LLM outputs generally (Topic 3's own explicit caveat)?
- **Monitoring:** tracking the judge's own sanity-check pass rate over time (does it continue to correctly distinguish known-good from known-bad cases in periodic validation samples) is itself a metric requiring Chapter 15 Topic 5's regression-tracking discipline — a judge that was well-validated once isn't guaranteed to remain reliable indefinitely, especially if the underlying model or prompt changes.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **How much sanity-checking investment is proportionate to the judge's actual role and stakes:** a judge informing a low-stakes, internal-only quality dashboard warrants less validation investment than one whose verdicts might inform actual deployment decisions or customer-facing escalation behavior — the right level of sanity-checking rigor should scale with the actual consequences of the judge being wrong, not be applied uniformly regardless of stakes.
- **Using a different, independent model for judging vs the convenience of reusing the same model:** an independent model reduces self-preference bias risk at the cost of managing an additional model integration — this trade-off should be weighed explicitly against the specific stakes involved, not decided by default convenience alone.
- **Periodic, sampled sanity-checking vs continuous, ongoing validation of every judge call:** continuous validation of every call is prohibitively expensive (requiring an independent verification for every judge call, largely negating the judge's own cost-saving value); periodic, sampled sanity-checking (this topic's recommended approach) balances practical cost against genuine, ongoing confidence in the judge's reliability.


### 6. Alternatives and When to Use Each

- **No sanity-checking at all, trusting judge verdicts unconditionally:** the approach this topic argues strongly against — reproduces exactly the "unverified model trust" risk this notebook series has repeatedly warned about for every other model-dependent behavior.
- **One-time sanity-checking, performed once when the judge is first built:** better than no checking at all, but insufficient given that judge reliability can drift over time (a model update, a prompt change) — Chapter 15 Topic 5's regression-tracking discipline should apply here too, not just a one-time validation.
- **Ongoing, periodic, sampled sanity-checking against human review (this topic's recommended default):** the right, sustainable balance between validation rigor and practical cost, especially appropriate given this project's regulated-domain context where judge-informed decisions carry real consequence.


### 7. Common Mistakes and Production Failures

- Trusting judge verdicts without any sanity-checking, reproducing the same "unverified model output" risk this notebook series has repeatedly warned against for every other model-dependent artifact.
- Using the exact same model for both the original agent task and the judging role by default, without considering the self-preference/correlated-blind-spot risk this specifically introduces.
- Validating a judge only against known-good cases (checking that it correctly passes them) without also validating against known-bad cases (checking that it correctly fails them), missing an overly lenient judge that provides false confidence.
- Performing judge sanity-checking only once, when the judge is first built, rather than as an ongoing, periodic practice that accounts for the possibility of drift over time.
- Not investigating a genuine disagreement between the judge's verdict and independent human review, assuming one is automatically correct rather than determining which actually was.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What is self-preference bias in the context of LLM-as-judge, and why is it a particular risk when the same model serves both roles?
  A: Self-preference bias is a judge model's tendency to systematically favor outputs resembling its own characteristic style or reasoning patterns, independent of genuine quality. This risk is particularly acute when the same model performs both the original task and the judging role, since a systematic weakness or stylistic tendency in the model's reasoning may go unrecognized by that same model acting as judge, precisely because the flaw exists in how the model reasons generally, not something it would flag in its own kind of output.

- Q: Why must a judge be validated against known-bad cases, not just known-good ones?
  A: A judge validated only against known-good cases could trivially "pass" validation by simply passing everything, including genuinely bad cases it should fail — this would produce an overly lenient judge providing false confidence that quality problems are being caught when they aren't. Validating against known-bad cases specifically confirms the judge can correctly identify genuine failures, not just correctly recognize genuine successes.

**Intermediate**

- Q: How does Topic 3's decomposed rubric approach help mitigate the surface-plausibility bias risk this topic identifies?
  A: A single, holistic "does this sound good" question is more easily swayed by confident tone or fluent phrasing, since it invites an overall, impressionistic judgment. Decomposed rubric criteria checking specific, semantic properties (does every cited fact actually appear in the source, does the stated conclusion follow from stated premises) force the judge's attention onto substantive, individually-checkable properties, directly reducing the risk that surface fluency alone drives a favorable verdict.

- Q: Why should judge sanity-checking be an ongoing, periodic practice rather than a one-time validation?
  A: A judge's reliability isn't fixed once validated — model updates, prompt changes, or shifts in the kinds of cases being judged could all cause a previously well-validated judge to become less reliable over time. This mirrors Chapter 15 Topic 5's regression-tracking discipline directly: a judge's sanity-check pass rate needs ongoing tracking, not a single, one-time confirmation assumed to hold indefinitely.

**Advanced**

- Q: Design an ongoing sanity-checking process for this project's reasoning-quality judge, balancing validation rigor against practical cost.
  A: Maintain a growing, curated set of known-good and known-bad reasoning-trace examples (starting with Topic 1's own two cases, expanded over time as new, clearly-classifiable examples are identified), and periodically re-run the judge against this full validation set — not just once, but on a regular schedule or whenever the underlying judge model or prompt changes — tracking the judge's accuracy against this known set over time using Chapter 15 Topic 5's regression-tracking discipline. Additionally, periodically sample a small subset of real, flagged cases (Topic 4's mechanism) for independent human review, specifically investigating any case where the judge and human reviewer disagree, using these investigations to both validate the judge's ongoing reliability and potentially expand the curated known-good/known-bad validation set with newly-discovered edge cases.

- Q: A teammate argues that since the judge was validated once and performed well, ongoing sanity-checking is unnecessary overhead. How do you respond?
  A: A one-time validation confirms the judge was reliable at that specific point in time, under those specific conditions — it provides no guarantee this reliability persists after a model update, a prompt change, or a shift in the kinds of real cases the judge encounters in ongoing use. This mirrors exactly why Chapter 15 Topic 5 argued for ongoing regression tracking rather than one-time evaluation for every other quantitative metric in this notebook series — a judge is not exempt from this same discipline simply because its output is qualitative rather than quantitative.

**Scenario-based**

- Q: During periodic sanity-checking, you find the judge and an independent human reviewer disagree on whether a specific case's reasoning trace passes the Factual Grounding criterion — the judge says it passes, the human reviewer says it fails due to a subtly invented claim the judge missed. How do you respond?
  A: This is a real, informative judge false-negative (a failure to catch — not to see, per se — but to correctly assess): investigate the specific case directly to confirm the human reviewer's assessment is correct (verifying the claim in question genuinely isn't supported by the source), then use this specific, concrete example to refine the rubric's Factual Grounding criterion description or worked examples (Topic 3's discipline) to better capture this specific kind of subtly-invented claim in the future, adding this case to the curated validation set so future judge sanity-checks specifically test for this now-known failure mode.

**Follow-up questions to expect:**

- "How would you distinguish a genuine judge unreliability problem from a case that's simply genuinely, legitimately ambiguous even for a careful human reviewer?"
- "What would you do if using an independent, different model for judging turned out to be significantly more expensive or slower than using the same model, given a project with real latency constraints?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **The specific risks this topic names (self-preference bias, surface-plausibility bias, position/framing sensitivity) are documented findings in published LLM-as-judge research generally**, not speculative concerns unique to this project — recognizing these as established, studied phenomena (rather than reinventing the concern from scratch) connects this topic's practical guidance to a broader, actively-researched area of LLM evaluation methodology.
- **The "who validates the validator" concern this topic raises is a specific instance of a very old, general epistemological and engineering problem**: any verification mechanism itself needs some form of validation, and that validation mechanism, followed to its logical extreme, could always need further validation — in practice, this infinite regress is resolved not by achieving some impossible, perfect certainty, but by grounding validation in independent, different mechanisms (human review, held-out known cases) at some practical, sufficient point, exactly this topic's actual recommended approach.
- **This topic closes Chapter 17's complete arc**: Topic 1 established why ground truth isn't enough for certain qualities, Topic 2 built the actual judge mechanism, Topic 3 formalized it into a documented rubric, Topic 4 applied it cost-consciously to real, flagged cases, and this topic closes the loop by ensuring the entire mechanism this chapter has built is itself trustworthy — completing a full, responsible LLM-as-judge practice rather than stopping at a working-but-unvalidated mechanism.

### 10. Quick Revision Summary (for last-minute interview prep)

> LLM-as-judge inherits every reliability concern this notebook series has raised about LLM outputs generally, requiring its own dedicated risk analysis and mitigation, not blind trust simply because it's positioned as an evaluation step. Specific, documented risks include self-preference bias (a judge favoring outputs resembling its own style, particularly acute when the same model serves both the original task and judging), surface-plausibility bias (being swayed by confident tone or fluent phrasing rather than genuine substance — directly mitigated by Topic 3's decomposed, semantic rubric criteria), position/framing sensitivity (verdicts shifting based on incidental presentation choices), and fundamental output inconsistency across repeated calls (since LLM outputs are inherently probabilistic). Sanity-checking a judge requires validating against both known-good and known-bad cases (not just known-good ones, which would let an overly lenient judge trivially "pass" validation), periodically cross-checking judge verdicts against independent human review with any disagreement specifically investigated, and treating this as an ongoing, ever-repeated practice — mirroring Chapter 15 Topic 5's regression-tracking discipline — rather than a one-time validation assumed to hold indefinitely as models, prompts, and real case distributions evolve over time. This closes Chapter 17's full arc: a complete, responsible LLM-as-judge practice includes not just building a working judge mechanism, but continuously verifying that mechanism itself remains trustworthy.


### Module 1: Sanity-Checking Against Known-Good AND Known-Bad Cases

Build an expanded validation set with BOTH known-good and known-bad reasoning traces, and test whether the judge correctly distinguishes them -- not just whether it passes the good ones.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Sanity-check against known-good AND known-bad cases
# ------------------------------------------------------------------

import re

def apply_factual_grounding_criterion(email: str, reasoning: str) -> bool:
    suspicious_claims = ["cancelled", "complaint", "angry", "rejected", "denied"]
    email_lower = email.lower()
    reasoning_lower = reasoning.lower()
    invented = [c for c in suspicious_claims if c in reasoning_lower and c not in email_lower]
    return len(invented) == 0

def apply_logical_coherence_criterion(reasoning: str, predicted_label: str) -> bool:
    reasoning_lower = reasoning.lower()
    label_lower = predicted_label.lower()
    clearly_claims = re.findall(r'clearly (?:a |an )?([a-z\-]+)', reasoning_lower)
    contradicts = any(c not in label_lower and label_lower not in c for c in clearly_claims)
    return not contradicts

def judge_reasoning(email: str, reasoning: str, predicted_label: str) -> bool:
    return apply_factual_grounding_criterion(email, reasoning) and apply_logical_coherence_criterion(reasoning, predicted_label)


# EXPANDED validation set -- BOTH known-good and known-bad cases,
# labeled by a human reviewer's ground-truth assessment of the REASONING
# ITSELF (not the final label) -- exactly this topic's requirement
VALIDATION_SET = [
    {"email": "What is the penalty for withdrawing my FD early?",
     "reasoning": "The email mentions FD and withdrawal penalty, clearly FD-related terms. No negation words present. Classifying as FD.",
     "predicted_label": "FD", "human_judgment": "GOOD"},
    {"email": "Is there a fee for early FD closure?",
     "reasoning": "The email asks about a fee for early closure of an FD, directly referencing FD terminology. Classifying as FD.",
     "predicted_label": "FD", "human_judgment": "GOOD"},
    {"email": "My loan is fine but I want to know about my FD interest rate too.",
     "reasoning": "This is clearly a loan complaint. However the customer's loan being cancelled and their obvious anger suggests Multiple Category, based on email length.",
     "predicted_label": "Multiple Category", "human_judgment": "BAD"},
    {"email": "Please tell me my account balance.",
     "reasoning": "The customer clearly filed a complaint about their rejected application, so this must be Non-FD due to the denial mentioned.",
     "predicted_label": "Non-FD", "human_judgment": "BAD"},  # invents "rejected"/"denial"/"complaint" not in email
]

print("=" * 70)
print("MODULE 1: SANITY-CHECKING AGAINST KNOWN-GOOD AND KNOWN-BAD CASES")
print("=" * 70)

true_positives = 0  # judge correctly passes a GOOD case
true_negatives = 0  # judge correctly fails a BAD case
false_positives = 0  # judge WRONGLY passes a BAD case (the dangerous, lenient-judge risk)
false_negatives = 0  # judge WRONGLY fails a GOOD case

for case in VALIDATION_SET:
    judge_verdict = judge_reasoning(case["email"], case["reasoning"], case["predicted_label"])
    human_says_good = case["human_judgment"] == "GOOD"

    if human_says_good and judge_verdict:
        true_positives += 1
    elif not human_says_good and not judge_verdict:
        true_negatives += 1
    elif not human_says_good and judge_verdict:
        false_positives += 1
    else:
        false_negatives += 1

    human_judgment = case["human_judgment"]
    verdict_label = "PASS" if judge_verdict else "FAIL"
    email_preview = case["email"][:55]
    print(f"\nHuman judgment: {human_judgment} | Judge verdict: {verdict_label}")
    print(f"  Email: '{email_preview}...'")

print(f"\nTrue positives (correctly passed GOOD cases): {true_positives}")
print(f"True negatives (correctly failed BAD cases): {true_negatives}")
print(f"False positives (WRONGLY passed BAD cases -- the DANGEROUS lenient-judge risk): {false_positives}")
print(f"False negatives (wrongly failed GOOD cases): {false_negatives}")

print(f"\nCONFIRMED: this validation specifically tests BOTH directions --")
print(f"not just 'does the judge pass good cases' but 'does the judge correctly")
print(f"REJECT bad cases too' -- exactly this topic's warning against an")
print(f"overly lenient judge that would trivially pass validation by")
print(f"approving everything.")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: SANITY-CHECKING AGAINST KNOWN-GOOD AND KNOWN-BAD CASES

Human judgment: GOOD | Judge verdict: PASS
  Email: 'What is the penalty for withdrawing my FD early?...'

Human judgment: GOOD | Judge verdict: PASS
  Email: 'Is there a fee for early FD closure?...'

Human judgment: BAD | Judge verdict: FAIL
  Email: 'My loan is fine but I want to know about my FD interest...'

Human judgment: BAD | Judge verdict: FAIL
  Email: 'Please tell me my account balance....'

True positives (correctly passed GOOD cases): 2
True negatives (correctly failed BAD cases): 2
False positives (WRONGLY passed BAD cases -- the DANGEROUS lenient-judge risk): 0
False negatives (wrongly failed GOOD cases): 0

CONFIRMED: this validation specifically tests BOTH directions --
not just 'does the judge pass good cases' but 'does the judge correctly
REJECT bad cases too' -- exactly this topic's warning against an
overly lenient judge that would trivially pass validation by
approving everything.

Module 1 complet

### Module 2: Demonstrating Self-Preference Bias Concretely

Simulate two judges (same-model-as-agent vs independent) evaluating the same reasoning trace, showing how a same-model judge can be systematically more lenient toward its own characteristic reasoning style.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Self-preference bias, demonstrated concretely
# ------------------------------------------------------------------

def same_model_judge(reasoning_trace: str) -> bool:
    """SIMULATES a judge using the SAME underlying model/style as the
    original agent -- honestly modeled here as being MORE LENIENT
    toward a specific stylistic pattern (verbose, hedged phrasing)
    that this same 'model family' characteristically produces,
    regardless of whether that verbosity reflects genuine quality."""
    verbose_hedging_markers = ["however", "it seems", "may", "based on", "appears"]
    hedge_count = sum(1 for marker in verbose_hedging_markers if marker in reasoning_trace.lower())
    # SIMULATED bias: a same-model judge is more likely to rate hedged,
    # verbose reasoning (its OWN characteristic style) as "thoughtful"
    # rather than genuinely uncertain/unfounded
    return hedge_count <= 3  # lenient threshold


def independent_model_judge(reasoning_trace: str, email_content: str) -> bool:
    """SIMULATES an INDEPENDENT judge model -- checks SUBSTANCE
    (Topic 3's actual fact-grounding logic) rather than being swayed
    by hedging language style at all."""
    return apply_factual_grounding_criterion(email_content, reasoning_trace)


test_case_email = "My loan is fine but I want to know about my FD interest rate too."
test_case_reasoning = (
    "This is clearly a loan complaint. However, the customer loan may be "
    "cancelled based on the tone."
)

same_model_verdict = same_model_judge(test_case_reasoning)
independent_verdict = independent_model_judge(test_case_reasoning, test_case_email)

print("=" * 70)
print("MODULE 2: SELF-PREFERENCE BIAS -- SAME-MODEL vs INDEPENDENT JUDGE")
print("=" * 70)
print(f"\nReasoning trace being judged: '{test_case_reasoning}'")
same_model_label = "PASS" if same_model_verdict else "FAIL"
independent_label = "PASS" if independent_verdict else "FAIL"
print(f"\nSame-model-style judge verdict (lenient toward hedged, verbose style): "
      f"{same_model_label}")
print(f"Independent judge verdict (checks substance -- fact-grounding): "
      f"{independent_label}")

if same_model_verdict and not independent_verdict:
    print(f"\nCONFIRMED: the same-model-style judge PASSED this reasoning trace,")
    print(f"swayed by its hedged phrasing STYLE (a surface quality), while the")
    print(f"independent judge, checking actual SUBSTANCE (does the reasoning")
    print(f"invent a 'cancelled' claim not in the source email), correctly")
    print(f"FAILED it. This is the CONCRETE, measurable self-preference/")
    print(f"surface-plausibility bias risk this topic's theory describes.")
else:
    print(f"\nIn this specific run, both judges reached the same verdict --")
    print(f"the bias this topic warns about did NOT manifest for this")
    print(f"particular case. This itself is an honest, useful finding: bias")
    print(f"risk is real and documented in LLM-as-judge research generally,")
    print(f"but doesn't manifest in EVERY case -- exactly why ongoing,")
    print(f"empirical sanity-checking across MANY cases (not a single")
    print(f"illustrative example) is what this topic's practice requires.")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: SELF-PREFERENCE BIAS -- SAME-MODEL vs INDEPENDENT JUDGE

Reasoning trace being judged: 'This is clearly a loan complaint. However, the customer loan may be cancelled based on the tone.'

Same-model-style judge verdict (lenient toward hedged, verbose style): PASS
Independent judge verdict (checks substance -- fact-grounding): FAIL

CONFIRMED: the same-model-style judge PASSED this reasoning trace,
swayed by its hedged phrasing STYLE (a surface quality), while the
independent judge, checking actual SUBSTANCE (does the reasoning
invent a 'cancelled' claim not in the source email), correctly
FAILED it. This is the CONCRETE, measurable self-preference/
surface-plausibility bias risk this topic's theory describes.

Module 2 complete. Run Module 3 next.


### Module 3: Consistency Across Superficial Presentation Changes

Test whether the judge's verdict changes based on incidental, superficial variations in how the SAME underlying case is presented -- a real, checkable robustness test.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Position/framing sensitivity -- consistency across
# superficial presentation changes
# ------------------------------------------------------------------

base_email = "My loan is fine but I want to know about my FD interest rate too."
base_reasoning = "This is clearly a loan complaint. The customer's loan being cancelled suggests Multiple Category."
base_label = "Multiple Category"

# SUPERFICIAL variations -- same underlying content, different incidental framing
presentation_variants = [
    (base_email, base_reasoning),  # original
    (base_email + " Thank you.", base_reasoning),  # trivial addition
    (base_email, "Reasoning: " + base_reasoning),  # trivial prefix
    (base_email.upper(), base_reasoning),  # different capitalization
]

print("=" * 70)
print("MODULE 3: JUDGE CONSISTENCY ACROSS SUPERFICIAL PRESENTATION CHANGES")
print("=" * 70)

verdicts = []
for i, (email_variant, reasoning_variant) in enumerate(presentation_variants):
    verdict = judge_reasoning(email_variant, reasoning_variant, base_label)
    verdicts.append(verdict)
    variant_label = "PASS" if verdict else "FAIL"
    print(f"\nVariant {i+1}: judge verdict = {variant_label}")
    print(f"  Email (truncated): '{email_variant[:50]}...'")

all_consistent = len(set(verdicts)) == 1
print(f"\nAll variants produced the SAME verdict despite superficial presentation")
print(f"differences: {all_consistent}")

if all_consistent:
    print(f"\nCONFIRMED: this specific judge implementation's verdict was NOT")
    print(f"swayed by trivial presentation changes (capitalization, incidental")
    print(f"prefixes/suffixes) -- a positive robustness signal for THIS")
    print(f"deterministic, rule-based stand-in. A REAL LLM judge would need this")
    print(f"SAME empirical test run against its actual behavior -- this kind of")
    print(f"position/framing sensitivity is a DOCUMENTED risk in real LLM-as-")
    print(f"judge research, and must be measured, never assumed away.")

print("\nModule 3 complete. All Chapter 17 Topic 5 modules done.")
print()
print("=" * 70)
print("CHAPTER 17 TOPIC 5 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Sanity-checking MUST test BOTH known-good AND known-bad cases --
  demonstrated concretely with a real true/false positive/negative
  breakdown, specifically catching the dangerous "overly lenient judge"
  risk that testing only good cases would miss entirely.

  Self-preference bias demonstrated CONCRETELY: a same-model-style judge
  (lenient toward hedged, verbose phrasing) PASSED a reasoning trace an
  independent, substance-checking judge correctly FAILED.

  Position/framing sensitivity tested EMPIRICALLY across superficial
  presentation variants -- this deterministic implementation stayed
  consistent, but a REAL LLM judge needs this SAME test run against its
  actual behavior, never assumed reliable without measurement.

  This closes Chapter 17's full arc: Topic 1 (why ground truth isn't
  enough), Topic 2 (build the judge), Topic 3 (formalize the rubric),
  Topic 4 (apply cost-consciously to real cases), Topic 5 (verify the
  judge mechanism itself is trustworthy) -- a COMPLETE, responsible
  LLM-as-judge practice.
""")


MODULE 3: JUDGE CONSISTENCY ACROSS SUPERFICIAL PRESENTATION CHANGES

Variant 1: judge verdict = FAIL
  Email (truncated): 'My loan is fine but I want to know about my FD int...'

Variant 2: judge verdict = FAIL
  Email (truncated): 'My loan is fine but I want to know about my FD int...'

Variant 3: judge verdict = FAIL
  Email (truncated): 'My loan is fine but I want to know about my FD int...'

Variant 4: judge verdict = FAIL
  Email (truncated): 'MY LOAN IS FINE BUT I WANT TO KNOW ABOUT MY FD INT...'

All variants produced the SAME verdict despite superficial presentation
differences: True

CONFIRMED: this specific judge implementation's verdict was NOT
swayed by trivial presentation changes (capitalization, incidental
prefixes/suffixes) -- a positive robustness signal for THIS
deterministic, rule-based stand-in. A REAL LLM judge would need this
SAME empirical test run against its actual behavior -- this kind of
position/framing sensitivity is a DOCUMENTED risk in real LLM-as-
judge 